# EXP-008: MLP 강화 (epochs 50→80, patience 15→20)

**근거:** EXP-007에서 MLP 모든 fold가 50 epoch 다 사용 (early stop 안 걸림). 학습 미수렴 가능성 → epochs↑ + patience↑로 자연 수렴 천장 확인.

**변경점 vs EXP-007:**
- MLP: MAX_EPOCHS 50 → **80**, patience 15 → **20**, T_max 50 → **80**
- 그 외 동일 (lr=1e-3, width 256-128-64, dropout 0.2, single seed)
- XGB/LGB/CAT는 EXP-007과 동일 (재학습으로 동일 split OOF 보장)

In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

TARGET = 'PitNextLap'
CAT_COLS = ['Driver', 'Compound', 'Race']
NUM_COLS = ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position',
            'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation',
            'RaceProgress', 'Position_Change']
y = train[TARGET].astype(int).values

train_le = train.copy(); test_le = test.copy()
for c in CAT_COLS:
    le = LabelEncoder()
    train_le[c] = le.fit_transform(train_le[c].astype(str))
    seen = set(le.classes_)
    test_le[c] = test_le[c].astype(str).map(lambda v: le.transform([v])[0] if v in seen else -1)

train_cb = train.copy(); test_cb = test.copy()
for c in CAT_COLS:
    train_cb[c] = train_cb[c].astype(str)
    test_cb[c]  = test_cb[c].astype(str)

drop_cols = ['id', TARGET]
feature_cols = [c for c in train.columns if c not in drop_cols]
X_le, X_test_le = train_le[feature_cols], test_le[feature_cols]
X_cb, X_test_cb = train_cb[feature_cols], test_cb[feature_cols]

N_FOLDS, SEED, LR = 5, 42, 0.02
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits = list(skf.split(X_le, y))
print(f'Train: {train.shape}, Test: {test.shape}')

Train: (439140, 16), Test: (188165, 15)


## 1. XGB (GPU, lr=0.02) — EXP-007 재현

In [2]:
from xgboost import XGBClassifier

xgb_params = dict(
    n_estimators=5000, max_depth=8, learning_rate=LR,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
    objective='binary:logistic', eval_metric='auc',
    device='cuda', tree_method='hist',
    random_state=SEED, verbosity=0,
    early_stopping_rounds=200,
)

oof_xgb = np.zeros(len(X_le))
test_xgb = np.zeros(len(X_test_le))
xgb_iters = []
t0 = time.time()
for fold, (tr_idx, va_idx) in enumerate(splits):
    m = XGBClassifier(**xgb_params)
    m.fit(X_le.iloc[tr_idx], y[tr_idx],
          eval_set=[(X_le.iloc[va_idx], y[va_idx])], verbose=0)
    oof_xgb[va_idx] = m.predict_proba(X_le.iloc[va_idx])[:, 1]
    test_xgb += m.predict_proba(X_test_le)[:, 1] / N_FOLDS
    xgb_iters.append(m.best_iteration)
    print(f'XGB fold {fold}: AUC={roc_auc_score(y[va_idx], oof_xgb[va_idx]):.5f}')
xgb_oof_auc = roc_auc_score(y, oof_xgb)
print(f'XGB OOF AUC: {xgb_oof_auc:.5f}  (best_iter mean {np.mean(xgb_iters):.0f}, elapsed {time.time()-t0:.1f}s)')

XGB fold 0: AUC=0.95072
XGB fold 1: AUC=0.94855
XGB fold 2: AUC=0.94935
XGB fold 3: AUC=0.94887
XGB fold 4: AUC=0.94988
XGB OOF AUC: 0.94946  (best_iter mean 2288, elapsed 69.4s)


## 2. LGB (CPU, lr=0.02) — EXP-007 재현

In [3]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

lgb_params = dict(
    n_estimators=5000, learning_rate=LR, num_leaves=64, max_depth=-1,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    objective='binary', metric='auc',
    random_state=SEED, n_jobs=-1, verbose=-1,
)

oof_lgb = np.zeros(len(X_le))
test_lgb = np.zeros(len(X_test_le))
lgb_iters = []
t0 = time.time()
for fold, (tr_idx, va_idx) in enumerate(splits):
    m = LGBMClassifier(**lgb_params)
    m.fit(X_le.iloc[tr_idx], y[tr_idx],
          eval_set=[(X_le.iloc[va_idx], y[va_idx])],
          callbacks=[early_stopping(200, verbose=False), log_evaluation(0)])
    oof_lgb[va_idx] = m.predict_proba(X_le.iloc[va_idx])[:, 1]
    test_lgb += m.predict_proba(X_test_le)[:, 1] / N_FOLDS
    lgb_iters.append(m.best_iteration_)
    print(f'LGB fold {fold}: AUC={roc_auc_score(y[va_idx], oof_lgb[va_idx]):.5f}')
lgb_oof_auc = roc_auc_score(y, oof_lgb)
print(f'LGB OOF AUC: {lgb_oof_auc:.5f}  (best_iter mean {np.mean(lgb_iters):.0f}, elapsed {time.time()-t0:.1f}s)')

LGB fold 0: AUC=0.95053
LGB fold 1: AUC=0.94847
LGB fold 2: AUC=0.94935
LGB fold 3: AUC=0.94860
LGB fold 4: AUC=0.94972
LGB OOF AUC: 0.94933  (best_iter mean 3927, elapsed 112.8s)


## 3. CatBoost (GPU, lr=0.02) — EXP-007 재현

In [4]:
from catboost import CatBoostClassifier

cat_idx = [feature_cols.index(c) for c in CAT_COLS]

cb_params = dict(
    iterations=20000, learning_rate=LR, depth=8, l2_leaf_reg=3.0,
    bagging_temperature=0.5, random_strength=1,
    eval_metric='AUC', loss_function='Logloss',
    cat_features=cat_idx, random_state=SEED,
    verbose=False, allow_writing_files=False,
    task_type='GPU', devices='0',
    early_stopping_rounds=200,
)

oof_cat = np.zeros(len(X_cb))
test_cat = np.zeros(len(X_test_cb))
cat_iters = []
t0 = time.time()
for fold, (tr_idx, va_idx) in enumerate(splits):
    m = CatBoostClassifier(**cb_params)
    m.fit(X_cb.iloc[tr_idx], y[tr_idx],
          eval_set=(X_cb.iloc[va_idx], y[va_idx]),
          use_best_model=True)
    oof_cat[va_idx] = m.predict_proba(X_cb.iloc[va_idx])[:, 1]
    test_cat += m.predict_proba(X_test_cb)[:, 1] / N_FOLDS
    cat_iters.append(m.get_best_iteration())
    print(f'CAT fold {fold}: AUC={roc_auc_score(y[va_idx], oof_cat[va_idx]):.5f}, best_iter={m.get_best_iteration()}')
cat_oof_auc = roc_auc_score(y, oof_cat)
print(f'CAT OOF AUC: {cat_oof_auc:.5f}  (best_iter mean {np.mean(cat_iters):.0f}, elapsed {time.time()-t0:.1f}s)')

Default metric period is 5 because AUC is/are not implemented for GPU


CAT fold 0: AUC=0.95067, best_iter=7401


Default metric period is 5 because AUC is/are not implemented for GPU


CAT fold 1: AUC=0.94860, best_iter=8526


Default metric period is 5 because AUC is/are not implemented for GPU


CAT fold 2: AUC=0.94953, best_iter=7324


Default metric period is 5 because AUC is/are not implemented for GPU


CAT fold 3: AUC=0.94896, best_iter=8628


Default metric period is 5 because AUC is/are not implemented for GPU


CAT fold 4: AUC=0.94982, best_iter=8418
CAT OOF AUC: 0.94951  (best_iter mean 8059, elapsed 1889.8s)


## 4. MLP 강화 (PyTorch GPU, MAX_EPOCHS 80, patience 20)

In [5]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Cat: -1 → 0 (unseen), 0..N-1 → 1..N (1+offset)
cat_cards = []
X_cat_tr = np.zeros((len(X_le), len(CAT_COLS)), dtype=np.int64)
X_cat_te = np.zeros((len(X_test_le), len(CAT_COLS)), dtype=np.int64)
for i, c in enumerate(CAT_COLS):
    n_unique = X_le[c].max() + 1
    cat_cards.append(n_unique + 1)
    X_cat_tr[:, i] = X_le[c].values + 1
    te_vals = X_test_le[c].values + 1
    te_vals[te_vals < 0] = 0
    X_cat_te[:, i] = te_vals

X_num_tr_raw = X_le[NUM_COLS].values.astype(np.float32)
X_num_te_raw = X_test_le[NUM_COLS].values.astype(np.float32)


class CatNumMLP(nn.Module):
    def __init__(self, cat_cards, emb_dims, n_numeric, hidden=(256, 128, 64), dropout=0.2):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(card, dim) for card, dim in zip(cat_cards, emb_dims)])
        emb_total = sum(emb_dims)
        self.bn_num = nn.BatchNorm1d(n_numeric)
        layers = []
        prev = emb_total + n_numeric
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.body = nn.Sequential(*layers)
    def forward(self, x_cat, x_num):
        emb = torch.cat([emb_layer(x_cat[:, i]) for i, emb_layer in enumerate(self.embs)], dim=1)
        x = torch.cat([emb, self.bn_num(x_num)], dim=1)
        return self.body(x).squeeze(1)

EMB_DIMS = [16, 4, 8]
BATCH_SIZE = 4096
MAX_EPOCHS = 80   # 50 → 80
PATIENCE = 20     # 15 → 20

oof_mlp = np.zeros(len(X_le))
test_mlp = np.zeros(len(X_test_le))
mlp_epochs_used = []
mlp_best_aucs = []
t0 = time.time()

for fold, (tr_idx, va_idx) in enumerate(splits):
    scaler = StandardScaler()
    X_num_tr = scaler.fit_transform(X_num_tr_raw[tr_idx])
    X_num_va = scaler.transform(X_num_tr_raw[va_idx])
    X_num_te = scaler.transform(X_num_te_raw)

    Xc_tr = torch.tensor(X_cat_tr[tr_idx], dtype=torch.long)
    Xn_tr = torch.tensor(X_num_tr, dtype=torch.float32)
    yt_tr = torch.tensor(y[tr_idx], dtype=torch.float32)
    Xc_va = torch.tensor(X_cat_tr[va_idx], dtype=torch.long, device=device)
    Xn_va = torch.tensor(X_num_va, dtype=torch.float32, device=device)
    yt_va = y[va_idx]
    Xc_te = torch.tensor(X_cat_te, dtype=torch.long, device=device)
    Xn_te = torch.tensor(X_num_te, dtype=torch.float32, device=device)

    train_loader = DataLoader(
        TensorDataset(Xc_tr, Xn_tr, yt_tr),
        batch_size=BATCH_SIZE, shuffle=True, drop_last=False,
        num_workers=0, pin_memory=True,
    )

    torch.manual_seed(SEED)
    model = CatNumMLP(cat_cards, EMB_DIMS, len(NUM_COLS)).to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=MAX_EPOCHS)
    bce = nn.BCEWithLogitsLoss()

    best_auc = -1
    best_state = None
    best_epoch = 0
    wait = 0
    epochs_used = 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        for xc, xn, yb in train_loader:
            xc = xc.to(device, non_blocking=True)
            xn = xn.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            optim.zero_grad()
            logit = model(xc, xn)
            loss = bce(logit, yb)
            loss.backward()
            optim.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            va_logit = model(Xc_va, Xn_va).cpu().numpy()
        va_prob = 1 / (1 + np.exp(-va_logit))
        auc = roc_auc_score(yt_va, va_prob)
        epochs_used = epoch + 1

        if auc > best_auc + 1e-6:
            best_auc = auc
            best_epoch = epochs_used
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        va_logit = model(Xc_va, Xn_va).cpu().numpy()
        te_logit = model(Xc_te, Xn_te).cpu().numpy()
    oof_mlp[va_idx] = 1 / (1 + np.exp(-va_logit))
    test_mlp += (1 / (1 + np.exp(-te_logit))) / N_FOLDS
    mlp_epochs_used.append(epochs_used)
    mlp_best_aucs.append(best_auc)
    early_stopped = epochs_used < MAX_EPOCHS
    print(f'MLP fold {fold}: best AUC={best_auc:.5f}, best_epoch={best_epoch}, epochs_used={epochs_used} {"(early stop)" if early_stopped else "(max)"}')

    del model, optim, sched, Xc_va, Xn_va, Xc_te, Xn_te
    torch.cuda.empty_cache()

mlp_oof_auc = roc_auc_score(y, oof_mlp)
print(f'MLP OOF AUC: {mlp_oof_auc:.5f}  (epochs mean {np.mean(mlp_epochs_used):.1f}, elapsed {time.time()-t0:.1f}s) [GPU]')
print(f'  vs EXP-007 MLP 0.94044: {mlp_oof_auc-0.94044:+.5f}')

MLP fold 0: best AUC=0.94251, best_epoch=79, epochs_used=80 (max)
MLP fold 1: best AUC=0.93995, best_epoch=70, epochs_used=80 (max)
MLP fold 2: best AUC=0.94051, best_epoch=65, epochs_used=80 (max)
MLP fold 3: best AUC=0.94047, best_epoch=79, epochs_used=80 (max)
MLP fold 4: best AUC=0.94137, best_epoch=76, epochs_used=80 (max)
MLP OOF AUC: 0.94095  (epochs mean 80.0, elapsed 1237.7s) [GPU]
  vs EXP-007 MLP 0.94044: +0.00051


## 5. Blending (4-way)

In [6]:
oof_blend_eq  = (oof_xgb + oof_lgb + oof_cat + oof_mlp) / 4
test_blend_eq = (test_xgb + test_lgb + test_cat + test_mlp) / 4
auc_eq = roc_auc_score(y, oof_blend_eq)

best = (None, -1)
for w_xgb in np.arange(0, 1.001, 0.05):
    for w_lgb in np.arange(0, 1.001 - w_xgb, 0.05):
        for w_cat in np.arange(0, 1.001 - w_xgb - w_lgb, 0.05):
            w_mlp = 1 - w_xgb - w_lgb - w_cat
            if w_mlp < 0: continue
            s = w_xgb*oof_xgb + w_lgb*oof_lgb + w_cat*oof_cat + w_mlp*oof_mlp
            a = roc_auc_score(y, s)
            if a > best[1]:
                best = ((round(w_xgb,2), round(w_lgb,2), round(w_cat,2), round(w_mlp,2)), a)
w_opt, auc_opt = best
oof_blend_opt  = w_opt[0]*oof_xgb + w_opt[1]*oof_lgb + w_opt[2]*oof_cat + w_opt[3]*oof_mlp
test_blend_opt = w_opt[0]*test_xgb + w_opt[1]*test_lgb + w_opt[2]*test_cat + w_opt[3]*test_mlp

best3 = (None, -1)
for w_xgb in np.arange(0, 1.001, 0.05):
    for w_lgb in np.arange(0, 1.001 - w_xgb, 0.05):
        w_cat = 1 - w_xgb - w_lgb
        if w_cat < 0: continue
        s = w_xgb*oof_xgb + w_lgb*oof_lgb + w_cat*oof_cat
        a = roc_auc_score(y, s)
        if a > best3[1]:
            best3 = ((round(w_xgb,2), round(w_lgb,2), round(w_cat,2)), a)

print('=== EXP-008 결과 (강화 MLP + 4-way) ===')
print(f'XGB OOF AUC: {xgb_oof_auc:.5f}')
print(f'LGB OOF AUC: {lgb_oof_auc:.5f}')
print(f'CAT OOF AUC: {cat_oof_auc:.5f}')
print(f'MLP OOF AUC: {mlp_oof_auc:.5f}  ← EXP-007 0.94044 대비 {mlp_oof_auc-0.94044:+.5f}')
print(f'Blend equal 4-way      : {auc_eq:.5f}')
print(f'Blend opt 4-way {w_opt}: {auc_opt:.5f}')
print(f'Blend opt 3-way (no MLP) {best3[0]}: {best3[1]:.5f}  ← MLP 효과 = {auc_opt - best3[1]:+.5f}')
print()
print('=== vs EXP-007 ===')
print(f'  Blend equal: 0.95020 -> {auc_eq:.5f}  ({auc_eq-0.95020:+.5f})')
print(f'  Blend opt:   0.95053 -> {auc_opt:.5f}  ({auc_opt-0.95053:+.5f})')
print(f'  MLP solo  : 0.94044 -> {mlp_oof_auc:.5f}  ({mlp_oof_auc-0.94044:+.5f})')

=== EXP-008 결과 (강화 MLP + 4-way) ===
XGB OOF AUC: 0.94946
LGB OOF AUC: 0.94933
CAT OOF AUC: 0.94951
MLP OOF AUC: 0.94095  ← EXP-007 0.94044 대비 +0.00051
Blend equal 4-way      : 0.95029
Blend opt 4-way (np.float64(0.25), np.float64(0.25), np.float64(0.4), np.float64(0.1)): 0.95053
Blend opt 3-way (no MLP) (np.float64(0.25), np.float64(0.3), np.float64(0.45)): 0.95041  ← MLP 효과 = +0.00011

=== vs EXP-007 ===
  Blend equal: 0.95020 -> 0.95029  (+0.00009)
  Blend opt:   0.95053 -> 0.95053  (-0.00000)
  MLP solo  : 0.94044 -> 0.94095  (+0.00051)


## 6. Submissions

In [7]:
for name, prob in [
    ('exp008_xgb',         test_xgb),
    ('exp008_lgb',         test_lgb),
    ('exp008_cat',         test_cat),
    ('exp008_mlp',         test_mlp),
    ('exp008_blend_equal', test_blend_eq),
    ('exp008_blend_opt',   test_blend_opt),
]:
    sub = pd.DataFrame({'id': test['id'], 'PitNextLap': prob})
    path = f'../submissions/submission_{name}.csv'
    sub.to_csv(path, index=False)
    print(f'  saved {path}  mean={prob.mean():.4f}')

  saved ../submissions/submission_exp008_xgb.csv  mean=0.1974
  saved ../submissions/submission_exp008_lgb.csv  mean=0.1972
  saved ../submissions/submission_exp008_cat.csv  mean=0.1980
  saved ../submissions/submission_exp008_mlp.csv  mean=0.2054
  saved ../submissions/submission_exp008_blend_equal.csv  mean=0.1995
  saved ../submissions/submission_exp008_blend_opt.csv  mean=0.1984
